# Análisis Exploratorio de Datos (EDA)
## Sistema de Inteligencia de Mercado - Don Piotr

Este notebook realiza el análisis exploratorio de los datos de restaurantes
extraídos de Google Maps, TripAdvisor y Bolivia en tus Manos para la ciudad de La Paz, Bolivia.

### Objetivos
1. Evaluar la calidad y completitud de los datos
2. Analizar distribuciones de variables clave
3. Identificar patrones geográficos y de mercado
4. Comparar fuentes de datos
5. Preparar recomendaciones para el módulo de Machine Learning

## 1. Setup y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

FIGURES_DIR = Path('../notebooks/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Librerías cargadas correctamente')

In [ ]:
# Cargar datos desde CSV o SQLite
DATA_DIR = Path('../scraping_don_piotr/output')

# Intentar cargar desde CSV primero
csv_path = DATA_DIR / 'restaurantes_la_paz.csv'
db_path = DATA_DIR / 'don_piotr.db'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f'Datos cargados desde CSV: {csv_path}')
elif db_path.exists():
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM restaurantes', conn)
    conn.close()
    print(f'Datos cargados desde SQLite: {db_path}')
else:
    raise FileNotFoundError(
        f'No se encontraron datos en {DATA_DIR}. '
        'Ejecute primero el scraping con main.py'
    )

print(f'\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')
df.head()

In [ ]:
# Información general del dataset
print('Tipos de datos:')
print(df.dtypes)
print(f'\nMemoria utilizada: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

## 2. Evaluación de Calidad de Datos

In [ ]:
# Análisis de valores nulos
null_counts = df.isnull().sum()
null_pct = (df.isnull().mean() * 100).round(1)

quality_df = pd.DataFrame({
    'Nulos': null_counts,
    'Porcentaje (%)': null_pct,
    'No Nulos': df.notna().sum(),
    'Tipo': df.dtypes
}).sort_values('Porcentaje (%)', ascending=False)

print('=== CALIDAD DE DATOS ===')
quality_df

In [ ]:
# Heatmap de valores nulos
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    df.isnull().T,
    cbar=True,
    cmap='YlOrRd',
    yticklabels=True,
    ax=ax
)
ax.set_title('Mapa de Valores Nulos por Campo', fontsize=16)
ax.set_xlabel('Registros', fontsize=12)
ax.set_ylabel('Campos', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nulls_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Completitud por fuente
if 'fuente' in df.columns:
    key_fields = ['nombre', 'direccion', 'telefono', 'rating',
                  'num_resenas', 'latitud', 'longitud', 'precio',
                  'tipo_cocina', 'zona']
    available_fields = [f for f in key_fields if f in df.columns]

    completeness_by_source = df.groupby('fuente')[available_fields].apply(
        lambda x: (x.notna().mean() * 100).round(1)
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    completeness_by_source.T.plot(kind='bar', ax=ax)
    ax.set_title('Completitud de Campos por Fuente (%)', fontsize=16)
    ax.set_xlabel('Campo', fontsize=12)
    ax.set_ylabel('Completitud (%)', fontsize=12)
    ax.set_ylim(0, 105)
    ax.legend(title='Fuente')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'completitud_por_fuente.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Análisis de duplicados
if 'nombre' in df.columns:
    exact_dupes = df.duplicated(subset=['nombre'], keep=False).sum()
    print(f'Registros con nombre duplicado (exacto): {exact_dupes}')

    if 'fuente' in df.columns:
        exact_dupes_source = df.duplicated(subset=['nombre', 'fuente'], keep=False).sum()
        print(f'Duplicados por nombre+fuente: {exact_dupes_source}')

    # Mostrar los nombres más repetidos
    name_counts = df['nombre'].value_counts()
    repeated = name_counts[name_counts > 1]
    if len(repeated) > 0:
        print(f'\nNombres que aparecen en múltiples registros ({len(repeated)}):')
        print(repeated.head(15))

## 3. Distribuciones Univariadas

In [ ]:
# Distribución de Rating
if 'rating' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histograma con KDE
    df['rating'].dropna().plot(
        kind='hist', bins=20, edgecolor='black', alpha=0.7,
        ax=axes[0], color='#2196F3'
    )
    axes[0].set_title('Distribución de Rating', fontsize=14)
    axes[0].set_xlabel('Rating (1-5)')
    axes[0].set_ylabel('Frecuencia')
    axes[0].axvline(df['rating'].mean(), color='red', linestyle='--',
                    label=f'Media: {df["rating"].mean():.2f}')
    axes[0].axvline(df['rating'].median(), color='green', linestyle='-.',
                    label=f'Mediana: {df["rating"].median():.2f}')
    axes[0].legend()

    # Box plot
    df.boxplot(column='rating', ax=axes[1])
    axes[1].set_title('Box Plot de Rating', fontsize=14)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'distribucion_rating.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nEstadísticas de Rating:')
    print(df['rating'].describe())

In [ ]:
# Distribución de Número de Reseñas (log-transformed)
if 'num_resenas' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Original
    df['num_resenas'].dropna().plot(
        kind='hist', bins=30, edgecolor='black', alpha=0.7,
        ax=axes[0], color='#FF9800'
    )
    axes[0].set_title('Distribución de Reseñas', fontsize=14)
    axes[0].set_xlabel('Número de Reseñas')
    axes[0].set_ylabel('Frecuencia')

    # Log-transformada
    resenas_log = np.log1p(df['num_resenas'].dropna())
    resenas_log.plot(
        kind='hist', bins=30, edgecolor='black', alpha=0.7,
        ax=axes[1], color='#FF9800'
    )
    axes[1].set_title('Distribución de Reseñas (Log)', fontsize=14)
    axes[1].set_xlabel('log(1 + Número de Reseñas)')
    axes[1].set_ylabel('Frecuencia')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'distribucion_resenas.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nEstadísticas de Reseñas:')
    print(df['num_resenas'].describe())

In [ ]:
# Distribución por Fuente y Zona
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Por fuente
if 'fuente' in df.columns:
    source_counts = df['fuente'].value_counts()
    colors_source = ['#2196F3', '#4CAF50', '#FF9800']
    source_counts.plot(
        kind='pie', autopct='%1.1f%%', startangle=90,
        colors=colors_source[:len(source_counts)],
        ax=axes[0]
    )
    axes[0].set_title('Distribución por Fuente', fontsize=14)
    axes[0].set_ylabel('')

# Por zona
if 'zona' in df.columns:
    zone_counts = df['zona'].dropna().value_counts()
    zone_counts.plot(
        kind='barh', color='#2196F3', edgecolor='black',
        alpha=0.7, ax=axes[1]
    )
    axes[1].set_title('Distribución por Zona', fontsize=14)
    axes[1].set_xlabel('Cantidad de Restaurantes')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'distribucion_fuente_zona.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribución por Tipo de Cocina (Top 20)
if 'tipo_cocina' in df.columns:
    # Separar tipos múltiples y contar
    all_cuisines = []
    for cuisines in df['tipo_cocina'].dropna():
        for c in str(cuisines).split(','):
            cleaned = c.strip()
            if cleaned:
                all_cuisines.append(cleaned)

    cuisine_series = pd.Series(all_cuisines)
    top_cuisines = cuisine_series.value_counts().head(20)

    fig, ax = plt.subplots(figsize=(12, 8))
    top_cuisines.plot(
        kind='barh', color='#4CAF50', edgecolor='black',
        alpha=0.7, ax=ax
    )
    ax.set_title('Top 20 Tipos de Cocina', fontsize=16)
    ax.set_xlabel('Cantidad de Restaurantes')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'top_cocinas.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Distribución por Nivel de Precio
if 'precio' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    price_order = ['$', '$$', '$$$', '$$$$']
    price_counts = df['precio'].dropna().value_counts()
    # Reordenar
    price_ordered = price_counts.reindex(price_order).dropna()

    price_ordered.plot(
        kind='bar', color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'],
        edgecolor='black', alpha=0.8, ax=ax
    )
    ax.set_title('Distribución por Nivel de Precio', fontsize=14)
    ax.set_xlabel('Rango de Precio')
    ax.set_ylabel('Cantidad de Restaurantes')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'distribucion_precio.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Análisis Bivariado

In [ ]:
# Rating vs Número de Reseñas
if 'rating' in df.columns and 'num_resenas' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))

    scatter_df = df.dropna(subset=['rating', 'num_resenas'])
    if 'fuente' in df.columns:
        for fuente, group in scatter_df.groupby('fuente'):
            ax.scatter(
                group['num_resenas'], group['rating'],
                label=fuente, alpha=0.6, s=40
            )
        ax.legend(title='Fuente')
    else:
        ax.scatter(
            scatter_df['num_resenas'], scatter_df['rating'],
            alpha=0.6, s=40, color='#2196F3'
        )

    ax.set_title('Rating vs Número de Reseñas', fontsize=16)
    ax.set_xlabel('Número de Reseñas')
    ax.set_ylabel('Rating')
    ax.set_xscale('symlog')  # Escala logarítmica para reseñas
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'rating_vs_resenas.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Rating por Zona (Box Plots)
if 'rating' in df.columns and 'zona' in df.columns:
    zone_data = df.dropna(subset=['rating', 'zona'])
    if len(zone_data) > 0:
        # Ordenar por mediana de rating
        zone_order = zone_data.groupby('zona')['rating'].median().sort_values(ascending=False).index

        fig, ax = plt.subplots(figsize=(14, 6))
        zone_data.boxplot(
            column='rating', by='zona', ax=ax,
            positions=range(len(zone_order)),
        )
        ax.set_xticklabels(zone_order, rotation=45, ha='right')
        ax.set_title('Distribución de Rating por Zona', fontsize=16)
        fig.suptitle('')  # Quitar título automático
        ax.set_xlabel('Zona')
        ax.set_ylabel('Rating')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'rating_por_zona.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# Rating por Tipo de Cocina (Violin Plots, Top 10)
if 'rating' in df.columns and 'tipo_cocina' in df.columns:
    # Tomar la primera cocina de cada restaurante
    df_cuisine = df.copy()
    df_cuisine['cocina_principal'] = df_cuisine['tipo_cocina'].apply(
        lambda x: str(x).split(',')[0].strip() if pd.notna(x) else None
    )

    # Top 10 cocinas por frecuencia
    top10 = df_cuisine['cocina_principal'].value_counts().head(10).index
    df_top = df_cuisine[df_cuisine['cocina_principal'].isin(top10)].dropna(subset=['rating'])

    if len(df_top) > 0:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.violinplot(
            data=df_top, x='cocina_principal', y='rating',
            order=top10, palette='Set2', ax=ax
        )
        ax.set_title('Distribución de Rating por Tipo de Cocina (Top 10)', fontsize=16)
        ax.set_xlabel('Tipo de Cocina')
        ax.set_ylabel('Rating')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'rating_por_cocina.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# Precio vs Rating
if 'precio' in df.columns and 'rating' in df.columns:
    price_rating = df.dropna(subset=['precio', 'rating'])
    if len(price_rating) > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        price_order = ['$', '$$', '$$$', '$$$$']
        available_prices = [p for p in price_order if p in price_rating['precio'].values]

        sns.boxplot(
            data=price_rating, x='precio', y='rating',
            order=available_prices, palette='RdYlGn', ax=ax
        )
        ax.set_title('Rating por Nivel de Precio', fontsize=16)
        ax.set_xlabel('Rango de Precio')
        ax.set_ylabel('Rating')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'precio_vs_rating.png', dpi=150, bbox_inches='tight')
        plt.show()

## 5. Análisis Geográfico

In [ ]:
# Mapa de restaurantes con coordenadas
geo_df = df.dropna(subset=['latitud', 'longitud'])
print(f'Restaurantes con coordenadas: {len(geo_df)} de {len(df)} ({len(geo_df)/len(df)*100:.1f}%)')

if len(geo_df) > 0:
    try:
        import folium
        from folium.plugins import HeatMap

        # Centro de La Paz
        center_lat = geo_df['latitud'].mean()
        center_lon = geo_df['longitud'].mean()

        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=13,
            tiles='OpenStreetMap'
        )

        # Color por rating
        def get_color(rating):
            if pd.isna(rating):
                return 'gray'
            if rating >= 4.5:
                return 'green'
            if rating >= 4.0:
                return 'blue'
            if rating >= 3.5:
                return 'orange'
            return 'red'

        for _, row in geo_df.iterrows():
            popup_text = f"{row.get('nombre', 'N/A')}<br>Rating: {row.get('rating', 'N/A')}"
            folium.CircleMarker(
                location=[row['latitud'], row['longitud']],
                radius=5,
                color=get_color(row.get('rating')),
                fill=True,
                fill_opacity=0.7,
                popup=popup_text
            ).add_to(m)

        # Heatmap de densidad
        heat_data = geo_df[['latitud', 'longitud']].values.tolist()
        HeatMap(heat_data, radius=15).add_to(m)

        display(m)
        m.save(str(FIGURES_DIR / 'mapa_restaurantes.html'))
        print('Mapa guardado en figures/mapa_restaurantes.html')

    except ImportError:
        print('folium no instalado. Generando scatter plot alternativo...')
        fig, ax = plt.subplots(figsize=(10, 10))
        scatter = ax.scatter(
            geo_df['longitud'], geo_df['latitud'],
            c=geo_df['rating'].fillna(0), cmap='RdYlGn',
            s=30, alpha=0.7, edgecolors='black', linewidth=0.5
        )
        plt.colorbar(scatter, label='Rating')
        ax.set_title('Mapa de Restaurantes en La Paz', fontsize=16)
        ax.set_xlabel('Longitud')
        ax.set_ylabel('Latitud')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'mapa_scatter.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# Cobertura por zona
if 'zona' in df.columns:
    from config import ZONAS_LA_PAZ

    zone_coverage = pd.DataFrame({
        'Zona': ZONAS_LA_PAZ,
        'Restaurantes': [df[df['zona'] == z].shape[0] for z in ZONAS_LA_PAZ],
        'Con Rating': [df[(df['zona'] == z) & df['rating'].notna()].shape[0] for z in ZONAS_LA_PAZ],
        'Con Coordenadas': [df[(df['zona'] == z) & df['latitud'].notna()].shape[0] for z in ZONAS_LA_PAZ],
    })
    zone_coverage = zone_coverage.sort_values('Restaurantes', ascending=False)

    print('=== COBERTURA POR ZONA ===')
    zone_coverage

## 6. Comparación de Fuentes

In [ ]:
# Análisis de overlap entre fuentes
if 'fuente' in df.columns and 'nombre' in df.columns:
    sources = df['fuente'].unique()
    print(f'Fuentes encontradas: {list(sources)}')
    print(f'Registros por fuente:')
    print(df['fuente'].value_counts())

    # Nombres normalizados por fuente
    source_names = {}
    for source in sources:
        names = set(
            df[df['fuente'] == source]['nombre']
            .str.lower().str.strip().dropna()
        )
        source_names[source] = names

    # Calcular overlap
    print('\n=== OVERLAP ENTRE FUENTES ===')
    for i, s1 in enumerate(sources):
        for s2 in sources[i+1:]:
            overlap = source_names[s1] & source_names[s2]
            print(f'{s1} ∩ {s2}: {len(overlap)} restaurantes en común')
            if overlap and len(overlap) <= 10:
                for name in sorted(overlap):
                    print(f'  - {name}')

In [ ]:
# Comparación de distribuciones de rating por fuente
if 'fuente' in df.columns and 'rating' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    for fuente in df['fuente'].unique():
        subset = df[df['fuente'] == fuente]['rating'].dropna()
        if len(subset) > 0:
            subset.plot(
                kind='kde', label=f'{fuente} (n={len(subset)})',
                ax=ax, linewidth=2
            )
    ax.set_title('Distribución de Rating por Fuente', fontsize=16)
    ax.set_xlabel('Rating')
    ax.set_ylabel('Densidad')
    ax.legend()
    ax.set_xlim(0, 5.5)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'rating_por_fuente.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Matriz de Correlación

In [ ]:
# Correlación de features numéricos
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Filtrar solo columnas relevantes
relevant_numeric = [c for c in numeric_cols if c not in ['data_quality']]

if len(relevant_numeric) >= 2:
    corr_matrix = df[relevant_numeric].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix, mask=mask, annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, square=True,
        linewidths=1, ax=ax
    )
    ax.set_title('Matriz de Correlación - Variables Numéricas', fontsize=16)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'correlacion_numerica.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Insuficientes columnas numéricas para calcular correlación')

## 8. Hallazgos Clave y Recomendaciones

In [ ]:
# Resumen estadístico general
print('=' * 60)
print('RESUMEN DEL ANÁLISIS EXPLORATORIO')
print('=' * 60)

print(f'\n1. VOLUMEN DE DATOS')
print(f'   Total de registros: {len(df)}')
if 'fuente' in df.columns:
    for fuente, count in df['fuente'].value_counts().items():
        print(f'   - {fuente}: {count}')

print(f'\n2. CALIDAD DE DATOS')
overall_completeness = (df.notna().mean() * 100).mean()
print(f'   Completitud promedio: {overall_completeness:.1f}%')
most_null = null_pct[null_pct > 0].sort_values(ascending=False)
if len(most_null) > 0:
    print(f'   Campos más incompletos:')
    for field, pct in most_null.head(5).items():
        print(f'   - {field}: {pct}% nulo')

if 'rating' in df.columns:
    print(f'\n3. RATINGS')
    print(f'   Rating promedio: {df["rating"].mean():.2f}')
    print(f'   Mediana: {df["rating"].median():.2f}')
    print(f'   Rango: {df["rating"].min():.1f} - {df["rating"].max():.1f}')

if 'zona' in df.columns:
    print(f'\n4. COBERTURA GEOGRÁFICA')
    zones_covered = df['zona'].dropna().nunique()
    print(f'   Zonas cubiertas: {zones_covered} de {len(config.ZONAS_LA_PAZ)}')
    top_zone = df['zona'].value_counts().index[0] if df['zona'].notna().any() else 'N/A'
    print(f'   Zona con más restaurantes: {top_zone}')

print(f'\n5. RECOMENDACIONES PARA ML')
print(f'   - Features numéricos sugeridos: rating, num_resenas, latitud, longitud')
print(f'   - Features categóricos sugeridos: tipo_cocina, zona, precio')
print(f'   - Imputar valores faltantes antes de clustering')
print(f'   - Estandarizar features numéricos (z-score)')
print(f'   - One-hot encoding para categóricos')